In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.metrics import roc_auc_score, average_precision_score

In [6]:
ANALYZE_DIR = Path('.').resolve()
REPO_DIR = ANALYZE_DIR.parent
OUTPUT_DIR = REPO_DIR / 'output'

simple_paths = [
    OUTPUT_DIR / 'history_3d_simple_cnn_1' / 'predict_val_3d_simple_cnn_1.csv',
    OUTPUT_DIR / 'history_3d_simple_cnn_2' / 'predict_val_3d_simple_cnn_2.csv',
    OUTPUT_DIR / 'history_3d_simple_cnn_3' / 'predict_val_3d_simple_cnn_3.csv',
    OUTPUT_DIR / 'history_3d_simple_cnn_4' / 'predict_val_3d_simple_cnn_4.csv',
    OUTPUT_DIR / 'history_3d_simple_cnn_5' / 'predict_val_3d_simple_cnn_5.csv',
]

residual_paths = [
    OUTPUT_DIR / 'history_3d_residual_cnn_160_pos_weight_1' / 'predict_val_3d_residual_cnn_160_pos_weight_1.csv',
    OUTPUT_DIR / 'history_3d_residual_cnn_160_pos_weight_2' / 'predict_val_3d_residual_cnn_160_pos_weight_2.csv',
    OUTPUT_DIR / 'history_3d_residual_cnn_160_pos_weight_3' / 'predict_val_3d_residual_cnn_160_pos_weight_3.csv',
    OUTPUT_DIR / 'history_3d_residual_cnn_160_pos_weight_4' / 'predict_val_3d_residual_cnn_160_pos_weight_4.csv',
    OUTPUT_DIR / 'history_3d_residual_cnn_160_pos_weight_5' / 'predict_val_3d_residual_cnn_160_pos_weight_5.csv',

]

print('Simple 3D files:')
for path in simple_paths:
    print(path)

print('\nResidual 3D files:')
for path in residual_paths:
    print(path)

Simple 3D files:
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_simple_cnn_1/predict_val_3d_simple_cnn_1.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_simple_cnn_2/predict_val_3d_simple_cnn_2.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_simple_cnn_3/predict_val_3d_simple_cnn_3.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_simple_cnn_4/predict_val_3d_simple_cnn_4.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_simple_cnn_5/predict_val_3d_simple_cnn_5.csv

Residual 3D files:
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_residual_cnn_160_pos_weight_1/predict_val_3d_residual_cnn_160_pos_weight_1.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_residual_cnn_160_pos_weight_2/predict_val_3d_residual_cnn_160_pos_weight_2.csv
/home/coder/project/jupyter/ich/ich-ct-classification/output/history_3d_resi

In [7]:
def read_and_clean_csv(path):
    df = pd.read_csv(path)

    df = df[['target', 'predict']].copy()

    for col in ['target', 'predict']:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(',', '.', regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['target', 'predict'])
    df = df[df['target'].isin([0, 1])]

    df['target'] = df['target'].astype(int)
    df['predict'] = df['predict'].astype(float)

    return df.reset_index(drop=True)

In [8]:
simple_df_1 = read_and_clean_csv(simple_paths[0])
simple_df_2 = read_and_clean_csv(simple_paths[1])
simple_df_3 = read_and_clean_csv(simple_paths[2])
simple_df_4 = read_and_clean_csv(simple_paths[3])
simple_df_5 = read_and_clean_csv(simple_paths[4])

residual_df_1 = read_and_clean_csv(residual_paths[0])
residual_df_2 = read_and_clean_csv(residual_paths[1])
residual_df_3 = read_and_clean_csv(residual_paths[2])
residual_df_4 = read_and_clean_csv(residual_paths[3])
residual_df_5 = read_and_clean_csv(residual_paths[4])

In [9]:
def mean_predictions(dfs):
    lengths = [len(df) for df in dfs]
    if len(set(lengths)) != 1:
        raise ValueError(f'Разные длины датафреймов: {lengths}')

    base_target = dfs[0]['target'].reset_index(drop=True)
    for i, df in enumerate(dfs[1:], start=2):
        cur_target = df['target'].reset_index(drop=True)
        if not base_target.equals(cur_target):
            raise ValueError(f'target не совпадает между dfs[0] и dfs[{i - 1}]')

    predict_mean = pd.concat(
        [df['predict'].reset_index(drop=True) for df in dfs],
        axis=1
    ).mean(axis=1)

    return pd.DataFrame({
        'target': base_target,
        'predict': predict_mean
    })


simple_mean_df = mean_predictions([
    simple_df_1,
    simple_df_2,
    simple_df_3,
    simple_df_4,
    simple_df_5,
])

residual_mean_df = mean_predictions([
    residual_df_1,
    residual_df_2,
    residual_df_3,
    residual_df_4,
    residual_df_5,
])

In [10]:
def bootstrap_roc_auc_delta(
    simple_df,
    residual_df,
    n_bootstrap: int = 2000,
    seed: int = 42
):
    rng = np.random.default_rng(seed)

    y_true = simple_df['target'].values
    p_simple = simple_df['predict'].values
    p_residual = residual_df['predict'].values

    n = len(y_true)
    deltas = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)

        y_b = y_true[idx]
        simple_b = p_simple[idx]
        residual_b = p_residual[idx]

        if len(np.unique(y_b)) < 2:
            continue

        roc_simple = roc_auc_score(y_b, simple_b)
        roc_residual = roc_auc_score(y_b, residual_b)

        deltas.append(roc_residual - roc_simple)

    deltas = np.array(deltas)
    ci = np.percentile(deltas, [2.5, 50, 97.5])

    results = {
        'mean': deltas.mean(),
        'ci_2.5': ci[0],
        'ci_50': ci[1],
        'ci_97.5': ci[2],
    }

    return results, deltas

In [11]:
def bootstrap_pr_auc_delta(
    simple_df,
    residual_df,
    n_bootstrap: int = 2000,
    seed: int = 42
):
    rng = np.random.default_rng(seed)

    y_true = simple_df['target'].values
    p_simple = simple_df['predict'].values
    p_residual = residual_df['predict'].values

    n = len(y_true)
    deltas = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)

        y_b = y_true[idx]
        simple_b = p_simple[idx]
        residual_b = p_residual[idx]

        if len(np.unique(y_b)) < 2:
            continue

        pr_simple = average_precision_score(y_b, simple_b)
        pr_residual = average_precision_score(y_b, residual_b)

        deltas.append(pr_residual - pr_simple)

    deltas = np.array(deltas)
    ci = np.percentile(deltas, [2.5, 50, 97.5])

    results = {
        'mean': round(deltas.mean(), 5),
        'ci_2.5': round(ci[0], 5),
        'ci_50': round(ci[1], 5),
        'ci_97.5': round(ci[2], 5),
    }

    return results, deltas

In [12]:
"""
Критерий:
CI полностью > 0 -> residual 3D улучшает
CI полностью < 0 -> residual 3D ухудшает
CI содержит 0 -> статистически незначимо
"""


def plot_bootstrap_deltas(
    deltas,
    metric_name: str = 'Metric Δ',
    nbins: int = 100,
    title: str | None = None,
    show_zero_line: bool = True,
    show_ci: bool = True
):
    deltas = np.asarray(deltas)

    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=deltas,
        nbinsx=nbins,
        name=metric_name,
        opacity=0.7
    ))

    if show_zero_line:
        fig.add_vline(x=0, line_dash='dash')

    if show_ci:
        ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
        fig.add_vline(x=ci_low, line_dash='dot')
        fig.add_vline(x=ci_high, line_dash='dot')

    fig.update_layout(
        title=title or f'{metric_name} Δ distribution',
        width=800,
        height=600,
        showlegend=False,
        xaxis_title=metric_name,
        yaxis_title='Count'
    )

    fig.show()

In [13]:
roc_res, roc_deltas = bootstrap_roc_auc_delta(simple_mean_df, residual_mean_df)
pr_res, pr_deltas = bootstrap_pr_auc_delta(simple_mean_df, residual_mean_df)

print('ROC:', roc_res)
print('PR:', pr_res)

plot_bootstrap_deltas(
    roc_deltas,
    metric_name='ROC AUC'
)

plot_bootstrap_deltas(
    pr_deltas,
    metric_name='PR AUC'
)

ROC: {'mean': np.float64(0.07392599466317534), 'ci_2.5': np.float64(-0.00037524386813166664), 'ci_50': np.float64(0.07418181818181813), 'ci_97.5': np.float64(0.14560709667416327)}
PR: {'mean': np.float64(0.09119), 'ci_2.5': np.float64(-0.00916), 'ci_50': np.float64(0.09014), 'ci_97.5': np.float64(0.18898)}
